# Enumeration recall ceiling — split AND merge errors reachable per hyperparameter setup

A **standalone diagnostic** (not part of the evolution loop): of all the split errors AND
all the merge errors that actually exist on a brain's GT, what fraction could a given
candidate-enumeration setup even PROPOSE? The evolved policy can only ever repair an error
that was enumerated up front (a `SplitSite` for a split error → `merge_labels`; a `MergeSite`
for a merge error → `split_label`), so this is the **hard ceiling** each enumeration setup
sets — the bound no accept/reject policy can beat.

Both enumerators are driven by the evolvable `ENUM_PARAMS` knobs (`resolve_enum_params`), so
this sweep answers the practical question: *which knob settings raise the reachable ceiling,
and at what candidate-count cost?* — the lever for closing an enumeration gap (as opposed to
a policy gap).

**Split-error denominator.** From `PreparedBrain._gt_raw` + GT graph adjacency, a *split
boundary* is an edge whose endpoints carry DIFFERENT nonzero segment ids (the segmentation
broke one neuron there). We count distinct adjacent label PAIRS per neuron (per-pair
denominator) and `needed = Σ(fragments−1)` per neuron (connectivity denominator).

**Merge-error denominator.** A raw label that lands (> `MIN` nodes) on ≥2 DISTINCT GT neurons
(the exact `%Merged Edges` / `detect_label_intersections` rule) — one label the segmentation
glued across two neurons; repairing it is a `split_label`. §2b ASSERTS this set equals the
package's own `labels_with_merge`, so the merge ceiling is benchmark-aligned.

**Cost structure (why the sweep is split in two).** `candidate_split_sites` is KD-tree
vectorized (~seconds–minutes), but `candidate_merge_sites` scans every branch node of the
whole-brain fragment graph (~tens of MINUTES per unique merge-knob tuple). So §3 defines the
setups + measures once, then **§3a runs the FAST split sweep** and **§3b runs the SLOW merge
sweep separately** — iterate the split ceiling freely without paying the merge scan; run §3b
deliberately (on a compute node). §3c/§4 combine whatever ran. Setups sharing enum params are
enumerated only once (the caches dedup). Reuses `candidate_split_sites` /
`candidate_merge_sites` so results tie out to the real generators. No image reads.

## 0. Setup — paths, brain, params

In [1]:
import os, sys, json, re
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJ = Path("../..").resolve()                      # exa-spim-agent/
sys.path.insert(0, str(PROJ))
sys.path.insert(0, str(PROJ.parent / "segmentation-skeleton-metrics" / "src"))

from proofreader_evolve.harness import incremental_scoring as inc
from proofreader_evolve.harness import dataset as ds

# --- parameters -----------------------------------------------------------------
# Point at a run to inherit its brain + mcl (so the ceiling matches what that run saw),
# OR set BRAIN_ID / MCL directly and leave RUN_NAME = None.
RUN_NAME = "794495_train1_test1_20260715_005246"
BRAIN_ID = None       # None -> auto from the run's split.json (TEST brain); else set e.g. "794491"
MCL      = None       # None -> auto from split.json ("mcl"); else set e.g. 100

RUNS = PROJ / "proofreader_evolve" / "runs"
if RUN_NAME:
    _cfg = json.loads((RUNS / RUN_NAME / "split.json").read_text())
    if BRAIN_ID is None:
        _t = _cfg.get("test_brains") or _cfg.get("brains") or []
        BRAIN_ID = str(_t[0]) if _t else None
    if MCL is None:
        MCL = int(_cfg.get("mcl", 100))
BRAIN_ID = str(BRAIN_ID); MCL = int(MCL or 100)
print(f"brain={BRAIN_ID}  mcl={MCL}  (from run {RUN_NAME})")

brain=794491  mcl=100  (from run 794495_train1_test1_20260715_005246)


## 1. Load prepared brain (GT labels) + fragment graph (for enumeration)

In [2]:
# PreparedBrain gives _gt_raw (per-node segment ids per GT skeleton) + gt_graphs.
_prep_path = Path(inc.shared_prepared_cache_path(BRAIN_ID))
try:
    prepared = inc.PreparedBrain.load_shared(str(_prep_path), expect_brain=BRAIN_ID)
    print(f"loaded SHARED prepared cache for {BRAIN_ID}")
except Exception:
    prepared = inc.PreparedBrain.load(str(_prep_path))
    print(f"loaded bare prepared pickle for {BRAIN_ID}")

# Fragment graph = what candidate_split_sites enumerates over (SAME mcl the run used).
cache_path = ds.default_cache_path(BRAIN_ID, min_cable_length=MCL)
fragments_graph, _gt, _payload = ds.load_cached_graphs(cache_path)
print(f"fragment graph: {fragments_graph.number_of_nodes()} nodes  ({cache_path})")

# label -> {neuron: node_count} over ALL GT (deployable 'which neuron does this fragment
# belong to' map); _dominant tells us a fragment label's home neuron.
label_gt_map = inc.label_gt_counts(prepared, gt_names=None)
def _dominant(lbl):
    return inc._dominant_neuron(label_gt_map, lbl)
print(f"{len(label_gt_map)} fragment labels land on some GT neuron")

loaded SHARED prepared cache for 794491
fragment graph: 31157765 nodes  (/allen/programs/mindscope/workgroups/auto-model/zihan.zhang/exaspim-agent/exa-spim-agent/cache/dataset_cache_794491_mcl100.pkl)
11162 fragment labels land on some GT neuron


## 2. Ground-truth denominators — split boundaries AND merge labels

Split: distinct adjacent label pairs per GT neuron (`gt_boundary_pairs`) + `needed =
Σ(fragments−1)`. Merge: raw labels spanning ≥2 GT neurons (≥`MIN` nodes each). Both are GT
truth, INDEPENDENT of what any setup enumerates — the honest ceiling denominators.

In [3]:
# --- SPLIT-error denominator: GT adjacency boundaries + connectivity ---------------
gt_boundary_pairs = set()          # {frozenset({A,B})} adjacent on some GT neuron
pair_example = {}                  # frozenset -> (neuron, xyz) representative boundary
neuron_labels = defaultdict(set)   # neuron -> set of segment ids on it
for name, g in prepared.gt_graphs.items():
    raw = prepared._gt_raw[name]                       # label per node index i
    for i, j in g.edges():
        li, lj = str(raw[i]), str(raw[j])
        if li != "0":
            neuron_labels[name].add(li)
        if lj != "0":
            neuron_labels[name].add(lj)
        if li != lj and li != "0" and lj != "0":
            pk = frozenset((li, lj))
            gt_boundary_pairs.add(pk)
            if pk not in pair_example:
                try:
                    xyz = tuple(round(float(v), 1) for v in g.node_xyz(i))
                except Exception:
                    xyz = None
                pair_example[pk] = (name, xyz)
# connectivity denominator: fragments-1 per multi-fragment neuron
needed = sum(max(len(labs) - 1, 0) for labs in neuron_labels.values())
n_frag_neurons = sum(1 for labs in neuron_labels.values() if len(labs) >= 2)

# --- MERGE-error denominator: raw labels spanning >=2 GT neurons (> MIN each) -------
# One segmentation label covering two real neurons; repairing it is a split_label. Uses
# the EXACT rule the benchmark's MergedEdgePercentMetric.detect_label_intersections uses:
# a label counts as a meaningful intersection only when it has STRICTLY > 50 nodes on
# EACH of >= 2 GT skeletons (note: strict >, not >=, to match the package byte-for-byte;
# the equivalence cell below asserts this set equals the package's own labels_with_merge).
MIN = inc._MERGE_MIN_NODES              # 50 (the metric's floor)
gt_merge_labels = {
    lab for lab, counts in label_gt_map.items()
    if sum(1 for c in counts.values() if c > MIN) >= 2
}

print(f"SPLIT denominators: {n_frag_neurons} multi-fragment neurons, "
      f"{len(gt_boundary_pairs)} boundary pairs, {needed} needed merges")
print(f"MERGE denominator : {len(gt_merge_labels)} labels spanning >=2 GT neurons "
      f"(> {MIN} nodes each)")

SPLIT denominators: 9 multi-fragment neurons, 7508 boundary pairs, 11368 needed merges
MERGE denominator : 85 labels spanning >=2 GT neurons (>= 50 nodes each)


## 2b. CONFIRM the merge denominator matches segmentation-skeleton-metrics

The merge denominator above is a REIMPLEMENTATION over raw GT node labels — it does NOT
call the metric package. So it could silently drift from the benchmark in three ways:
(1) the threshold boundary (this notebook now uses strict `> 50` to match), (2) a label on
3+ neurons (pooled-count vs the package's pairwise-graph intersection), and (3) the
misalignment fix (`_gt_raw` is snapshotted BEFORE `fix_label_misalignments`; the package
scores AFTER). This cell removes the doubt: it runs the package's OWN
`MergedEdgePercentMetric.detect_label_intersections` on the SAME post-fix GT graphs the
scorer builds, collects the union of `labels_with_merge` (the benchmark's authoritative
merge-label set), and ASSERTS it equals `gt_merge_labels`. Cheap — pure graph work, NOT the
~30-min `evaluate()`. A mismatch prints the offending labels and WHICH of the three causes
it is, instead of failing silently downstream.

In [ ]:
# Build the GT graphs the way the scorer/evaluate() does: identity LabelHandler +
# fix_label_misalignments (this is exactly PreparedBrain._apply_handler with no edits),
# then run the PACKAGE'S OWN merge detector and compare its label set to ours.
from segmentation_skeleton_metrics.skeleton_metrics import MergedEdgePercentMetric

_pkg_handler = inc.LabelHandler()                 # identity: get(raw) -> raw segment id
prepared._apply_handler(_pkg_handler)             # sets node_label = raw, then fix_label_misalignments()
_merge_metric = MergedEdgePercentMetric(verbose=False)
_merge_metric(prepared.gt_graphs)                 # populates each graph's labels_with_merge

# Authoritative benchmark merge-label set = union of labels_with_merge across all GT graphs.
pkg_merge_labels = set()
for _g in prepared.gt_graphs.values():
    pkg_merge_labels |= {str(l) for l in getattr(_g, "labels_with_merge", set())}

_only_ours = gt_merge_labels - pkg_merge_labels
_only_pkg  = pkg_merge_labels - gt_merge_labels
print(f"our gt_merge_labels : {len(gt_merge_labels)}")
print(f"package labels_with_merge: {len(pkg_merge_labels)}")
print(f"agree on: {len(gt_merge_labels & pkg_merge_labels)}   "
      f"only-ours: {len(_only_ours)}   only-package: {len(_only_pkg)}")

def _why(lab):
    # Classify a disagreeing label by the three known divergence causes.
    c = label_gt_map.get(str(lab), {})
    over = [n for n, k in c.items() if k > MIN]
    at50 = [n for n, k in c.items() if k == MIN]        # threshold-boundary (strict > vs >=)
    n_over = len(over)
    if len(at50) and n_over < 2:
        return f"threshold-boundary (neuron(s) at exactly {MIN} nodes: {at50})"
    if n_over >= 3:
        return f"3+-neuron label (pooled-vs-pairwise; over-threshold on {n_over} neurons)"
    return "misalignment-fix (raw vs post-fix labels differ near boundary nodes)"

if _only_ours or _only_pkg:
    print("\nMISMATCH — divergence detail:")
    for lab in sorted(_only_ours):
        print(f"  only ours    {lab}: {_why(lab)}  counts={label_gt_map.get(str(lab), {})}")
    for lab in sorted(_only_pkg):
        print(f"  only package {lab}: {label_gt_map.get(str(lab), {})}  "
              f"(package saw it post-misalignment-fix; raw snapshot did not)")
    print("\n[note] a small mismatch here is EXPECTED at the misalignment boundary and does "
          "not invalidate the ceiling; it quantifies how far the raw-label proxy sits from "
          "the benchmark. Investigate only if the count is large.")
else:
    print("\n[OK] merge denominator is byte-for-byte the benchmark's merge-label set "
          "(detect_label_intersections). The merge ceiling is benchmark-aligned.")

# The apply_handler call above relabeled the GT graphs in place; re-snapshot nothing is
# needed (label_gt_map / _gt_raw are from the raw snapshot and unchanged), but any later
# cell that reads prepared.gt_graphs[...].node_label sees the identity relabeling, which
# is what the scorer uses anyway. gt_boundary_pairs / neuron_labels were built from
# _gt_raw (raw), so they are unaffected.

## 3. Hyperparameter setups + recall measures (shared defs — cheap)

Each SETUP names a split-enum knob set AND a merge-enum knob set. This cell only DEFINES
them + the measure functions + per-tuple enum caches — no enumeration runs here. The SPLIT
sweep (§3a, ~seconds–minutes) and the MERGE sweep (§3b, ~tens of MINUTES, whole-brain scan)
are separate cells below, so you can run the fast split ceiling repeatedly WITHOUT paying
the merge scan. Each unique param tuple is enumerated once (the caches dedup).

In [ ]:
import time

# --- SPLIT recall of one enumerated SplitSite set --------------------------------
def _uf_new(): return {}
def _find(uf, x):
    uf.setdefault(x, x)
    while uf[x] != x:
        uf[x] = uf[uf[x]]; x = uf[x]
    return x
def _union(uf, a, b):
    ra, rb = _find(uf, a), _find(uf, b)
    if ra != rb: uf[ra] = rb

def measure_split(sites):
    enum_pairs, same_neuron, uf = set(), 0, _uf_new()
    for s in sites:
        a, b = str(s.label_a), str(s.label_b)
        enum_pairs.add(frozenset((a, b)))
        da, db = _dominant(a), _dominant(b)
        if da is not None and da == db:          # a REAL (same-neuron) edge
            same_neuron += 1; _union(uf, a, b)
    hit = len(gt_boundary_pairs & enum_pairs)
    per_pair = hit / len(gt_boundary_pairs) if gt_boundary_pairs else float("nan")
    reachable = 0
    for name, labs in neuron_labels.items():
        if len(labs) < 2: continue
        comps = len({_find(uf, l) for l in labs})  # union-find over REAL edges
        reachable += len(labs) - comps
    conn = reachable / needed if needed else float("nan")
    prec = same_neuron / len(sites) if sites else float("nan")
    return dict(n_split_cand=len(sites), split_pairs_hit=hit,
                split_per_pair_recall=per_pair, split_conn_recall=conn,
                split_precision=prec)

# --- MERGE recall of one enumerated MergeSite set --------------------------------
def measure_merge(merge_sites):
    detected = {str(s.label) for s in merge_sites}      # distinct labels a MergeSite hit
    reachable = gt_merge_labels & detected
    recall = len(reachable) / len(gt_merge_labels) if gt_merge_labels else float("nan")
    prec = (len(reachable) / len(detected)) if detected else float("nan")  # detected that are real
    return dict(n_merge_cand=len(merge_sites), n_merge_labels=len(detected),
                merge_reachable=len(reachable), merge_recall=recall, merge_precision=prec)

# --- setups: each varies BOTH split and merge enumeration knobs ------------------
# split knobs: max_gap_um, per_tip_k, max_sites, tip_to_shaft
# merge knobs: min_arm_cable_um, seed_depth_um, max_sites, max_per_label
SETUPS = [
    dict(name="defaults",
         split=dict(max_gap_um=15, per_tip_k=4,  max_sites=5000,  tip_to_shaft=True),
         merge=dict(min_arm_cable_um=10, seed_depth_um=8, max_sites=5000,  max_per_label=8)),
    dict(name="wide split",
         split=dict(max_gap_um=40, per_tip_k=32, max_sites=50000, tip_to_shaft=True),
         merge=dict(min_arm_cable_um=10, seed_depth_um=8, max_sites=5000,  max_per_label=8)),
    dict(name="wide merge",
         split=dict(max_gap_um=15, per_tip_k=4,  max_sites=5000,  tip_to_shaft=True),
         merge=dict(min_arm_cable_um=4,  seed_depth_um=6, max_sites=50000, max_per_label=32)),
    dict(name="wide both",
         split=dict(max_gap_um=40, per_tip_k=32, max_sites=50000, tip_to_shaft=True),
         merge=dict(min_arm_cable_um=4,  seed_depth_um=6, max_sites=50000, max_per_label=32)),
]

# Per-unique-tuple enum caches (so setups sharing params enumerate once).
_split_cache, _merge_cache = {}, {}
def _get_split(p):
    k = (p["max_gap_um"], p["per_tip_k"], p["max_sites"], p["tip_to_shaft"])
    if k not in _split_cache:
        t = time.monotonic()
        sites, _ = ds.candidate_split_sites(
            fragments_graph, max_gap_um=p["max_gap_um"], max_sites=p["max_sites"],
            per_tip_k=p["per_tip_k"], tip_to_shaft=p["tip_to_shaft"], return_stats=True)
        print(f"  split enum {k}: {len(sites)} sites ({time.monotonic()-t:.0f}s)", flush=True)
        _split_cache[k] = sites
    return _split_cache[k]
def _get_merge(p):
    k = (p["min_arm_cable_um"], p["seed_depth_um"], p["max_sites"], p["max_per_label"])
    if k not in _merge_cache:
        t = time.monotonic()
        sites = ds.candidate_merge_sites(
            fragments_graph, min_arm_cable_um=p["min_arm_cable_um"],
            seed_depth_um=p["seed_depth_um"], max_sites=p["max_sites"],
            max_per_label=p["max_per_label"])
        print(f"  merge enum {k}: {len(sites)} sites ({time.monotonic()-t:.0f}s)", flush=True)
        _merge_cache[k] = sites
    return _merge_cache[k]

n_uniq_split = len({(s["split"]["max_gap_um"], s["split"]["per_tip_k"],
                     s["split"]["max_sites"], s["split"]["tip_to_shaft"]) for s in SETUPS})
n_uniq_merge = len({(s["merge"]["min_arm_cable_um"], s["merge"]["seed_depth_um"],
                     s["merge"]["max_sites"], s["merge"]["max_per_label"]) for s in SETUPS})
print(f"{len(SETUPS)} setups -> {n_uniq_split} unique SPLIT enum(s) (fast) + "
      f"{n_uniq_merge} unique MERGE enum(s) (SLOW whole-brain scan, ~tens of min each)")

## 3a. SPLIT-error recall sweep (FAST — run this freely)

Enumerates SplitSites for each unique split-knob tuple and measures split recall. No merge
scan, so this is cheap (~seconds–minutes) — iterate `SETUPS['split']` here without paying
the merge cost.

In [ ]:
rows_split = []
for st in SETUPS:
    m = {"setup": st["name"]}
    m.update(measure_split(_get_split(st["split"])))
    rows_split.append(m)
res_split = pd.DataFrame(rows_split).set_index("setup")
for c in ["split_per_pair_recall", "split_conn_recall", "split_precision"]:
    res_split[c] = (100 * res_split[c]).round(1)
print(f"SPLIT denom: {len(gt_boundary_pairs)} boundary pairs / {needed} needed merges")
res_split[["n_split_cand", "split_pairs_hit", "split_per_pair_recall",
           "split_conn_recall", "split_precision"]]

## 3b. MERGE-error recall sweep (SLOW — whole-brain scan, run on a compute node)

Enumerates MergeSites for each unique merge-knob tuple. `candidate_merge_sites` scans every
branch node of the whole-brain fragment graph, so each UNIQUE merge tuple is ~tens of
minutes. Run deliberately (ideally on n251). Skip this cell if you only need the split
ceiling — the combined table (§3c) and plots (§4) handle merge results being absent.

In [ ]:
rows_merge = []
for st in SETUPS:
    print(f"setup: {st['name']} (merge) ...", flush=True)
    m = {"setup": st["name"]}
    m.update(measure_merge(_get_merge(st["merge"])))
    rows_merge.append(m)
res_merge = pd.DataFrame(rows_merge).set_index("setup")
for c in ["merge_recall", "merge_precision"]:
    res_merge[c] = (100 * res_merge[c]).round(1)
print(f"MERGE denom: {len(gt_merge_labels)} labels spanning >=2 GT neurons")
res_merge[["n_merge_cand", "merge_reachable", "merge_recall", "merge_precision"]]

## 3c. Combined table — split & merge recall per setup

Joins whatever ran above. If §3b (merge) was skipped, only the split columns show.

In [ ]:
# Join split (§3a) and merge (§3b) results; tolerate either being absent.
_frames = []
if "res_split" in dir(): _frames.append(res_split)
if "res_merge" in dir(): _frames.append(res_merge)
assert _frames, "run §3a (split) and/or §3b (merge) first"
res = pd.concat(_frames, axis=1)
res = res.loc[:, ~res.columns.duplicated()]      # 'setup' index shared; drop dup cols
_cols = [c for c in ["n_split_cand", "split_pairs_hit", "split_per_pair_recall",
                     "split_conn_recall", "split_precision", "n_merge_cand",
                     "merge_reachable", "merge_recall", "merge_precision"]
         if c in res.columns]
res[_cols]

## 4. Plot split & merge recall (and enumeration cost) across setups

In [ ]:
# Plots whatever ran: split bars always; merge bars only if §3b was run (res has the
# merge columns). So this cell works after a fast split-only pass too.
_has_merge = "merge_recall" in res.columns
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
x = np.arange(len(res)); w = 0.25 if _has_merge else 0.4
ax[0].bar(x - (w if _has_merge else w/2), res["split_per_pair_recall"], w,
          label="split per-pair", color="#1f77b4")
ax[0].bar(x if _has_merge else x + w/2, res["split_conn_recall"], w,
          label="split connectivity", color="#2ca02c")
if _has_merge:
    ax[0].bar(x + w, res["merge_recall"], w, label="merge recall", color="#d62728")
ax[0].set_xticks(x); ax[0].set_xticklabels(res.index, rotation=25, ha="right", fontsize=8)
ax[0].set_ylabel("% of GT errors reachable"); ax[0].set_ylim(0, 100)
ax[0].set_title("Recall ceiling by setup — split"
                + (" vs merge" if _has_merge else " (merge sweep §3b not run)"))
ax[0].legend(fontsize=8); ax[0].grid(True, axis="y", alpha=0.3)

if _has_merge:
    ax[1].bar(x - w/2, res["n_split_cand"], w, label="# split cand", color="#1f77b4")
    ax[1].bar(x + w/2, res["n_merge_cand"], w, label="# merge cand", color="#d62728")
else:
    ax[1].bar(x, res["n_split_cand"], 0.5, label="# split cand", color="#1f77b4")
ax[1].set_yscale("log"); ax[1].set_xticks(x)
ax[1].set_xticklabels(res.index, rotation=25, ha="right", fontsize=8)
ax[1].set_ylabel("# candidates (log; cost / noise)")
ax[1].set_title("Enumeration cost by setup")
ax[1].legend(fontsize=8); ax[1].grid(True, axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 5. How to read this

- **split per-pair recall** — did the setup propose a candidate on the exact `{A,B}` GT
  boundary? **split connectivity recall** — union-find over enumerated same-neuron edges:
  can a chain of merges make each broken neuron whole? (matches the loop's split-repair
  ceiling). **merge recall** — of labels spanning ≥2 GT neurons, how many did a `MergeSite`
  surface (the only labels a `split_label` can ever act on).
- **These are ceilings, not policy scores.** `recall = reachable / all-GT` caps ANY policy;
  the shortfall is an ENUMERATION gap — closable only by widening `ENUM_PARAMS` (the knobs
  swept here) or adding a detector, NOT by the accept/reject policy. The precision columns
  are the cost: wider enumeration reaches more but dilutes the stream, so the policy must
  work harder to reject the extra non-errors.
- **Use it to choose `ENUM_PARAMS`:** pick the knee where recall is high but candidate count
  (cost/noise) is still manageable, then set those knobs in the policy's `ENUM_PARAMS`.
- Split and merge are duals — a two-phase run needs BOTH ceilings high (split_label needs
  reachable merge errors; merge_labels needs reachable split errors).